# KBO 2022~2026 데이터 전처리 & 가을야구 진출 예측
**모델**: Logistic + Random Forest + Gradient Boosting 앙상블  
**데이터**: 2022(완료) · 2023(완료) · 2024(완료) · 2025(완료) · 2026(진행중)  
**기준일**: 2026-04-24 | 팀당 약 22경기 진행  

| CELL | 내용 |
|------|------|
| 1 | 라이브러리 & 환경 설정 |
| 2 | 전처리 유틸리티 함수 |
| 3 | 카테고리별 전처리 함수 (12종) |
| 4 | 데이터 로드 (processed CSV → data dict) |
| 5 | 인사이트 분석 테이블 생성 (마스터 JOIN) |
| 6 | 실시간 크롤링 (2026 자정 자동 업데이트) |
| 7 | 탐색적 데이터 분석 EDA |
| 8 | 가을야구 진출 예측 (앙상블 ML) |
| 9 | 심층 분석 — 4월 예측력 · 팀 클러스터링 · 선수이동 영향 |
| 10 | 최종 결론 출력 |

## CELL 1 · 라이브러리 & 환경 설정

In [1]:
# !pip install pandas numpy scikit-learn matplotlib seaborn requests beautifulsoup4 schedule
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings, platform, os, time
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, LeaveOneGroupOut
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score

# ── 한글 폰트 ──────────────────────────────────────────────────
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    try:
        fp = [f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name][0]
        plt.rcParams['font.family'] = fm.FontProperties(fname=fp).get_name()
    except:
        plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print("✅ 라이브러리 OK")
print(f"   pandas {pd.__version__} / numpy {np.__version__}")

✅ 라이브러리 OK
   pandas 3.0.2 / numpy 2.4.4


## CELL 2 · 전처리 유틸리티 함수

In [2]:
TEAM_NAME_MAP = {
    "SK":"SSG","SK와이번스":"SSG","SSG랜더스":"SSG",
    "KIA타이거즈":"KIA","LG트윈스":"LG","KT위즈":"KT",
    "두산베어스":"두산","삼성라이온즈":"삼성","롯데자이언츠":"롯데",
    "NC다이노스":"NC","키움히어로즈":"키움","한화이글스":"한화",
    "히어로즈":"키움","넥센":"키움",
}

def read_csv_safe(path):
    for enc in ['utf-8-sig','utf-8','cp949','euc-kr']:
        try: return pd.read_csv(path, encoding=enc)
        except: continue
    raise ValueError(f"읽기 실패: {path}")

def normalize_team(name):
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

def parse_ip(s):
    """'173 1/3' → 173.333""";s=str(s).strip()
    if s in ['-','','nan','NaN']: return np.nan
    if '/' in s:
        parts=s.split(); w=int(parts[0]) if len(parts)>1 else 0
        n,d=parts[-1].split('/'); return round(w+int(n)/int(d),4)
    try: return float(s)
    except: return np.nan

def parse_dash(df, cols):
    """'-' 문자 → NaN 변환 후 숫자 타입 변환""";df=df.copy()
    for c in cols:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).replace('-',np.nan),errors='coerce')
    return df

def parse_record(s):   return s.str.extract(r'(\d+)-(\d+)-(\d+)').astype(float)
def parse_r10(s):       return s.str.extract(r'(\d+)승(\d+)무(\d+)패').astype(float)

def pythagorean_wp(rs, ra, exp=1.83):
    rs,ra=np.array(rs,float),np.array(ra,float)
    return np.where((rs>0)&(ra>0), rs**exp/(rs**exp+ra**exp), np.nan)

# ── 파생 지표 ──────────────────────────────────────────────────
def bat_drv(df):
    """타자 파생 지표 추가""";df=df.copy()
    if all(c in df.columns for c in ['HR','AB']):
        df['HR/AB']=(df['HR']/df['AB'].replace(0,np.nan)).round(4)
    if all(c in df.columns for c in ['H','HR','AB','SO','SF']):
        denom=df['AB']-df['SO']-df['HR']+df['SF']
        df['BABIP']=((df['H']-df['HR'])/denom.replace(0,np.nan)).round(3)
    if all(c in df.columns for c in ['BB','SO']):
        df['BB/K']=(df['BB']/df['SO'].replace(0,np.nan)).round(3)
    if all(c in df.columns for c in ['SLG','AVG']):
        df['ISO']=(df['SLG']-df['AVG']).round(3)
    if all(c in df.columns for c in ['BB','HBP','H','2B','3B','HR','PA']):
        df['wOBA']=((0.69*df['BB']+0.72*df['HBP']
                     +0.89*(df['H']-df['2B']-df['3B']-df['HR'])
                     +1.27*df['2B']+1.62*df['3B']+2.10*df['HR'])
                    /df['PA'].replace(0,np.nan)).round(3)
    return df

def pit_drv(df):
    """투수 파생 지표 추가""";df=df.copy()
    if all(c in df.columns for c in ['SO','IP','BB','HR','HBP']):
        ip=df['IP'].replace(0,np.nan)
        df['K/9']   =(df['SO'] /ip*9).round(2)
        df['BB/9']  =(df['BB'] /ip*9).round(2)
        df['HR/9']  =(df['HR'] /ip*9).round(2)
        df['K/BB']  =(df['SO'] /df['BB'].replace(0,np.nan)).round(2)
        df['FIP']   =((13*df['HR']+3*(df['BB']+df['HBP'])-2*df['SO'])/ip+3.2).round(3)
    return df

print("✅ 유틸리티 함수 OK")

✅ 유틸리티 함수 OK


## CELL 3 · 카테고리별 전처리 함수 (12종)

In [3]:
# ── 선수 기록 ────────────────────────────────────────────────
def f_bat_b(df, y):
    """타자 기본기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df=parse_dash(df,['AVG','G','PA','AB','R','H','2B','3B','HR','TB','RBI',
                      'SAC','SF','BB','IBB','HBP','SO','GDP','SLG','OBP','OPS','MH','RISP'])
    df=bat_drv(df)
    thr=int(22*3.1) if y==2026 else 450          # 2026 비례 규정타석
    df['규정타석_충족']=df['PA']>=thr
    df['시즌진행중']=(y==2026); return df

def f_bat_d(df, y):
    """타자 세부기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    return parse_dash(df,['AVG','XBH','GO','AO','GO/AO','GW RBI','BB/K','P/PA','ISOP','XR','GPA'])

def f_pit_b(df, y):
    """투수 기본기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df['IP']=df['IP'].apply(parse_ip)
    df=parse_dash(df,['ERA','G','W','L','SV','HLD','WPCT','H','HR','BB','HBP','SO','R','ER','WHIP'])
    df=pit_drv(df)
    thr=22 if y==2026 else 144                   # 2026 비례 규정이닝
    df['규정이닝_충족']=df['IP']>=thr
    df['시즌진행중']=(y==2026); return df

def f_pit_d(df, y):
    """투수 세부기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    return parse_dash(df,['ERA','GS','Wgs','Wgr','GF','SVO','TS','GDP','GO','AO','GO/AO'])

def f_fld(df, y):
    """수비 기본기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df['IP']=df['IP'].apply(parse_ip)
    df=parse_dash(df,['G','GS','E','PKO','PO','A','DP','FPCT','PB','SB','CS'])
    df['CS%']=pd.to_numeric(df['CS%'].replace('-',np.nan),errors='coerce')
    pos_map={'투수':'P','포수':'C','1루수':'1B','2루수':'2B','3루수':'3B',
             '유격수':'SS','좌익수':'LF','중견수':'CF','우익수':'RF'}
    if 'POS' in df.columns: df['POS_EN']=df['POS'].map(pos_map)
    return df

def f_run(df, y):
    """주루 기본기록""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df=parse_dash(df,['G','SBA','SB','CS','SB%','OOB','PKO'])
    df['SB%_calc']=(df['SB']/df['SBA'].replace(0,np.nan)*100).round(1); return df

# ── 팀 기록 ────────────────────────────────────────────────────
def f_trk(df, y):
    """팀 순위""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df=parse_dash(df,['순위','경기','승','패','무','승률','게임차'])
    h=parse_record(df['홈']);   h.columns=['홈승','홈무','홈패']
    v=parse_record(df['방문']); v.columns=['원정승','원정무','원정패']
    r=parse_r10(df['최근10경기']); r.columns=['최근승','최근무','최근패']
    df=pd.concat([df.drop(columns=['홈','방문','최근10경기','연속'],errors='ignore'),h,v,r],axis=1)
    df['홈승률']   =(df['홈승']  /(df['홈승']  +df['홈패']  )).round(3)
    df['원정승률'] =(df['원정승']/(df['원정승']+df['원정패'])).round(3)
    df['홈_원정_승률차']=(df['홈승률']-df['원정승률']).round(3)
    df['가을야구']=(df['순위']<=5).astype(int) if y<2026 else np.nan
    df['시즌진행중']=(y==2026); return df

def f_dly(df, y):
    """팀 일자별순위""";df=df.copy(); df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    df['날짜']=pd.to_datetime(df['날짜'].astype(str),format='%Y%m%d')
    df['월']=df['날짜'].dt.month; df['연도_실제']=df['날짜'].dt.year
    df=parse_dash(df,['순위','경기','승','패','무','승률','게임차'])
    r=parse_r10(df['최근10경기']); r.columns=['최근승','최근무','최근패']
    h=parse_record(df['홈']);       h.columns=['홈승','홈무','홈패']
    v=parse_record(df['방문']);     v.columns=['원정승','원정무','원정패']
    df=pd.concat([df.drop(columns=['최근10경기','연속','홈','방문'],errors='ignore'),r,h,v],axis=1)
    df=df.sort_values(['팀명','날짜'])
    df['순위_변동']=df.groupby('팀명')['순위'].diff().fillna(0)*-1; return df

def f_tg(df, y, cat):
    """팀 공통 (타자/투수/수비/주루)""";df=df.copy()
    df['연도']=y; df['팀명']=df['팀명'].apply(normalize_team)
    if 'IP'  in df.columns: df['IP']=df['IP'].apply(parse_ip)
    if 'CS%' in df.columns: df['CS%']=pd.to_numeric(df['CS%'].replace('-',np.nan),errors='coerce')
    skip={'팀명','연도','CS%','POS'}
    for col in df.columns:
        if col in skip: continue
        if df[col].dtype==object:
            df[col]=pd.to_numeric(df[col].astype(str).replace('-',np.nan),errors='coerce')
    if cat=='팀_투수' and all(c in df.columns for c in ['SO','IP','BB','HR','HBP']):
        df=pit_drv(df)
    if cat=='팀_타자' and 'G' in df.columns:
        if 'HR' in df.columns: df['HR/G']=(df['HR']/df['G'].replace(0,np.nan)).round(3)
        if 'R'  in df.columns: df['득점/G']=(df['R'] /df['G'].replace(0,np.nan)).round(3)
        df=bat_drv(df)
    if cat=='팀_주루':
        df['SB%_calc']=(df['SB']/df['SBA'].replace(0,np.nan)*100).round(1)
    return df

FILE_PROC_MAP = {
    '타자_기본기록':    f_bat_b,
    '타자_세부기록':    f_bat_d,
    '투수_기본기록':    f_pit_b,
    '투수_세부기록':    f_pit_d,
    '수비_기본기록':    f_fld,
    '주루_기본기록':    f_run,
    '팀_순위':          f_trk,
    '팀_일자별순위':    f_dly,
    '팀_타자_기본기록': lambda df,y: f_tg(df,y,'팀_타자'),
    '팀_투수_기본기록': lambda df,y: f_tg(df,y,'팀_투수'),
    '팀_수비_기본기록': lambda df,y: f_tg(df,y,'팀_수비'),
    '팀_주루_기본기록': lambda df,y: f_tg(df,y,'팀_주루'),
}
print("✅ 전처리 함수 OK (12종)")

✅ 전처리 함수 OK (12종)


## CELL 4 · 데이터 로드
> **경로 설정**: `RAW_ROOT` 를 연도별 폴더가 있는 경로로 수정하세요.  
> processed CSV가 있으면 바로 로드, 없으면 원본 재전처리합니다.

In [4]:
# ── 경로 설정 ──────────────────────────────────────────────────
RAW_ROOT      = Path('./data/raw')         # 원본: ./data/raw/2022/, 2023/, ...
PROCESSED_DIR = Path('./data/processed')   # 통합 CSV 폴더
YEARS         = [2022, 2023, 2024, 2025, 2026]
COMPLETE_YRS  = [2022, 2023, 2024, 2025]  # 시즌 완료 연도
PLAYOFF_CUT   = 5                          # 포스트시즌 진출 기준

data = {}   # ★ 모든 분석의 시작점

# ── 통합 CSV 우선 로드 ─────────────────────────────────────────
FILE_KEY_MAP = {
    '01_team_rank':         '팀_순위',
    '02_team_bat':          '팀_타자_기본기록',
    '03_team_pit':          '팀_투수_기본기록',
    '04_team_def':          '팀_수비_기본기록',
    '05_team_run':          '팀_주루_기본기록',
    '06_daily_rank':        '팀_일자별순위',
    '07_player_bat':        '타자_기본기록',
    '08_player_bat_detail': '타자_세부기록',
    '09_player_pit':        '투수_기본기록',
    '10_player_pit_detail': '투수_세부기록',
    '11_player_def':        '수비_기본기록',
    '12_player_run':        '주루_기본기록',
    'master_transfer':      '선수이동현황',
}

if PROCESSED_DIR.exists():
    print(f"📂 통합 CSV 로드: {PROCESSED_DIR.resolve()}")
    for fname, key in FILE_KEY_MAP.items():
        p = PROCESSED_DIR / f'{fname}.csv'
        if p.exists():
            df = read_csv_safe(str(p))
            if '날짜' in df.columns:
                df['날짜'] = pd.to_datetime(df['날짜'], errors='coerce')
            data[key] = df
            print(f"  ✔ '{key}': {df.shape}  연도: {sorted(df['연도'].unique().tolist()) if '연도' in df.columns else '-'}")
else:
    print(f"⚠️  processed 폴더 없음 → 원본 연도별 재전처리")
    for year in YEARS:
        year_dir = RAW_ROOT / str(year)
        if not year_dir.exists():
            print(f"   [{year}] 폴더 없음 — 스킵"); continue
        print(f"── {year}년 처리 중...")
        for key, func in FILE_PROC_MAP.items():
            fpath = year_dir / f'{key}.csv'
            if not fpath.exists(): continue
            df = func(read_csv_safe(str(fpath)), year)
            data[key] = pd.concat([data[key], df], ignore_index=True) if key in data else df

print(f"\n{'='*55}")
print(f"✅ data 딕셔너리: {len(data)}개 키")
print(f"   {list(data.keys())}")
required = ['타자_기본기록','투수_기본기록','팀_순위','팀_일자별순위',
            '팀_타자_기본기록','팀_투수_기본기록']
missing = [k for k in required if k not in data]
if missing: print(f"\n❌ 누락 키: {missing}")
else:        print("\n✅ 필수 키 모두 존재 → 다음 셀 실행 가능")
print('='*55)

⚠️  processed 폴더 없음 → 원본 연도별 재전처리
   [2022] 폴더 없음 — 스킵
   [2023] 폴더 없음 — 스킵
   [2024] 폴더 없음 — 스킵
   [2025] 폴더 없음 — 스킵
   [2026] 폴더 없음 — 스킵

✅ data 딕셔너리: 0개 키
   []

❌ 누락 키: ['타자_기본기록', '투수_기본기록', '팀_순위', '팀_일자별순위', '팀_타자_기본기록', '팀_투수_기본기록']


## CELL 5 · 인사이트 분석 테이블 생성

In [5]:
assert 'data' in dir() and len(data)>0, "CELL 4를 먼저 실행하세요!"

# ── 5-1. 타자 마스터 (기본 + 세부) ───────────────────────────
bat_b = data['타자_기본기록'].copy()
bat_d = data['타자_세부기록'].copy()
batter_master = pd.merge(
    bat_b,
    bat_d.drop(columns=['AVG','순위'], errors='ignore'),
    on=['선수명','팀명','연도'], how='left'
)
print(f"타자_마스터: {batter_master.shape}")
display(batter_master[batter_master['연도']==2026].head(3))

# ── 5-2. 투수 마스터 (기본 + 세부) ───────────────────────────
pit_b = data['투수_기본기록'].copy()
pit_d = data['투수_세부기록'].copy()
pitcher_master = pd.merge(
    pit_b,
    pit_d.drop(columns=['ERA','순위'], errors='ignore'),
    on=['선수명','팀명','연도'], how='left'
)
print(f"\n투수_마스터: {pitcher_master.shape}")
display(pitcher_master[pitcher_master['연도']==2026].head(3))

# ── 5-3. 팀 종합 마스터 ──────────────────────────────────────
# 팀순위를 베이스로 타자·투수·수비·주루 JOIN
team_master = data['팀_순위'].copy()

for tk, sf in [('팀_타자_기본기록','타자'), ('팀_투수_기본기록','투수'),
               ('팀_수비_기본기록','수비'), ('팀_주루_기본기록','주루')]:
    tdf  = data[tk].copy()
    excl = {'팀명','연도','순위','G','경기'}
    cols = [c for c in tdf.columns if c not in excl and c not in team_master.columns]
    m    = tdf[['팀명','연도']+cols].rename(columns={c: f'{c}_{sf}' for c in cols})
    team_master = pd.merge(team_master, m, on=['팀명','연도'], how='left')

# 파생 지표
team_master['가을야구'] = (team_master['순위']<=PLAYOFF_CUT).astype(int)
team_master.loc[team_master['연도']==2026,'가을야구'] = np.nan   # 진행중

bat_r = team_master.get('R_타자', team_master.get('득점/G_타자', pd.Series(np.nan, index=team_master.index)) * team_master.get('경기', pd.Series(np.nan, index=team_master.index)))
pit_r = team_master.get('R_투수', pd.Series(np.nan, index=team_master.index))

# 득실차·기대승률: 팀별 R 컬럼 자동 탐지
bat_R_col = next((c for c in team_master.columns if c.startswith('R_') and '타자' in c), None)
pit_R_col = next((c for c in team_master.columns if c.startswith('R_') and '투수' in c), None)
if bat_R_col and pit_R_col:
    team_master['득실차']   = team_master[bat_R_col] - team_master[pit_R_col]
    rs, ra = team_master[bat_R_col].values.astype(float), team_master[pit_R_col].values.astype(float)
    team_master['기대승률'] = np.where((rs>0)&(ra>0), rs**1.83/(rs**1.83+ra**1.83), np.nan)
    team_master['승률_운']  = (team_master['승률'] - team_master['기대승률']).round(4)

print(f"\n팀_종합_마스터: {team_master.shape}")
display(team_master[team_master['연도']==2026].head(3))

# ── 5-4. 일자별 통계 집계 ────────────────────────────────────
daily = data['팀_일자별순위'].copy()
daily['날짜'] = pd.to_datetime(daily['날짜'], errors='coerce')
daily['월'] = daily['날짜'].dt.month

# 순위 변동성 (팀별)
rank_vol = (daily.groupby(['연도','팀명'])['순위']
            .agg(['mean','std','min','max'])
            .reset_index())
rank_vol.columns = ['연도','팀명','평균순위','순위변동성','최고순위','최저순위']
rank_vol['순위변동성'] = rank_vol['순위변동성'].round(2)

# 월별 평균순위
monthly_rank = (daily.groupby(['연도','팀명','월'])['순위']
                .mean().round(2).reset_index()
                .rename(columns={'순위':'월평균순위'}))

# ── 5-5. 리그 환경 지표 ──────────────────────────────────────
규정타자 = bat_b[bat_b.get('규정타석_충족', pd.Series(True, index=bat_b.index))==True]
규정투수 = pit_b[pit_b.get('규정이닝_충족', pd.Series(True, index=pit_b.index))==True]

league_bat = 규정타자.groupby('연도')[['AVG','OPS','HR','RBI','BABIP']].mean().round(3).reset_index()
league_pit = 규정투수.groupby('연도')[['ERA','WHIP','K/9','BB/9','FIP']].mean().round(3).reset_index()

print("\n=== 리그 타격환경 (규정타자 평균) ===")
display(league_bat)
print("=== 리그 투구환경 (규정투수 평균) ===")
display(league_pit)
print("\n✅ 인사이트 테이블 생성 완료")

AssertionError: CELL 4를 먼저 실행하세요!

## CELL 6 · 실시간 크롤링 (2026 자정 자동 업데이트)
> **자정 스케줄**: 주석 해제 후 실행하면 매일 00:00 자동 갱신  
> **즉시 실행**: `crawl_all()` 호출

In [ ]:
import requests, schedule
from bs4 import BeautifulSoup
from datetime import datetime
import urllib3; urllib3.disable_warnings()

BASE_URL = 'https://www.koreabaseball.com'
CUR_YEAR = datetime.now().year
HEADERS  = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36'),
    'Referer': 'https://www.koreabaseball.com',
    'Accept-Language': 'ko-KR,ko;q=0.9',
}

def get_page(url, params=None):
    r = requests.get(url, headers=HEADERS, params=params, verify=False, timeout=15)
    r.raise_for_status(); r.encoding='utf-8'
    return BeautifulSoup(r.text, 'html.parser')

def parse_table(soup, idx=0):
    tables = soup.find_all('table')
    if not tables or idx>=len(tables): return pd.DataFrame()
    t = tables[idx]
    hdrs = [th.get_text(strip=True) for th in t.find_all('th')]
    rows = [[td.get_text(strip=True) for td in tr.find_all('td')]
            for tr in t.find_all('tr')[1:] if tr.find_all('td')]
    if not rows: return pd.DataFrame()
    mc = max(len(r) for r in rows)
    hdrs = (hdrs[:mc] if len(hdrs)>=mc
            else hdrs+[f'col_{i}' for i in range(len(hdrs),mc)])
    return pd.DataFrame(rows, columns=hdrs)

# ── 크롤링 함수 ────────────────────────────────────────────────
def crawl_team_rank(year=CUR_YEAR):
    soup = get_page(f'{BASE_URL}/Record/Team/TeamRank/teamRankDetail.aspx',
                    {'leagueId':'1','srId':'0','seasonId':str(year)})
    df=parse_table(soup); df['연도']=year; return df

def crawl_daily_rank(year=CUR_YEAR):
    soup = get_page(f'{BASE_URL}/Record/Team/TeamRank/teamRankDaily.aspx',
                    {'leagueId':'1','srId':'0','seasonId':str(year)})
    df=parse_table(soup); df['연도']=year; return df

def crawl_player(stat, detail=False, year=CUR_YEAR):
    m = {'batter': ('Batter','BatterBasic'  if not detail else 'BatterDetail'),
         'pitcher':('Pitcher','PitcherBasic' if not detail else 'PitcherDetail'),
         'fielder':('Fielder','FielderBasic'),
         'runner': ('Runner', 'RunnerBasic')}
    cat, page = m[stat]
    soup = get_page(f'{BASE_URL}/Record/Player/{cat}/{page}.aspx',
                    {'leagueId':'1','srId':'0','seasonId':str(year),
                     'teamId':'','positionId':'','sortKey':'','orderBy':'DESC',
                     'pageNo':'1','numOfRows':'200'})
    df=parse_table(soup); df['연도']=year; return df

def crawl_team_stat(stat, year=CUR_YEAR):
    m = {'batter': 'TeamBatterBasic','pitcher':'TeamPitcherBasic',
         'fielder':'TeamFielderBasic','runner': 'TeamRunnerBasic'}
    soup = get_page(f'{BASE_URL}/Record/Team/{m[stat]}.aspx',
                    {'leagueId':'1','srId':'0','seasonId':str(year)})
    df=parse_table(soup); df['연도']=year; return df

def crawl_transfer(year=CUR_YEAR):
    """선수 이동현황 크롤링""";df_list=[]
    for page_no in range(1,6):
        try:
            soup=get_page(f'{BASE_URL}/Player/Transfer/List.aspx',
                          {'seasonId':str(year),'pageNo':str(page_no)})
            df=parse_table(soup)
            if df.empty: break
            df_list.append(df)
        except: break
    return pd.concat(df_list,ignore_index=True) if df_list else pd.DataFrame()

# ── 전체 크롤링 실행 ───────────────────────────────────────────
def crawl_all(save_dir=Path('./data/raw/2026'), year=CUR_YEAR):
    """2026 전체 크롤링 → processed 갱신""";save_dir.mkdir(parents=True,exist_ok=True)
    jobs = [
        ('팀_순위',          lambda: crawl_team_rank(year)),
        ('팀_일자별순위',    lambda: crawl_daily_rank(year)),
        ('타자_기본기록',    lambda: crawl_player('batter',False,year)),
        ('타자_세부기록',    lambda: crawl_player('batter',True, year)),
        ('투수_기본기록',    lambda: crawl_player('pitcher',False,year)),
        ('투수_세부기록',    lambda: crawl_player('pitcher',True, year)),
        ('수비_기본기록',    lambda: crawl_player('fielder',False,year)),
        ('주루_기본기록',    lambda: crawl_player('runner', False,year)),
        ('팀_타자_기본기록', lambda: crawl_team_stat('batter', year)),
        ('팀_투수_기본기록', lambda: crawl_team_stat('pitcher',year)),
        ('팀_수비_기본기록', lambda: crawl_team_stat('fielder',year)),
        ('팀_주루_기본기록', lambda: crawl_team_stat('runner', year)),
    ]
    results = {}
    for name, func in jobs:
        try:
            df=func()
            df.to_csv(save_dir/f'{name}.csv',index=False,encoding='utf-8-sig')
            if name in FILE_PROC_MAP:
                processed = FILE_PROC_MAP[name](df, year)
                # data 딕셔너리에서 해당 연도 교체
                if name in data:
                    data[name] = pd.concat([
                        data[name][data[name]['연도']!=year],
                        processed
                    ], ignore_index=True)
                else:
                    data[name] = processed
            results[name] = df.shape
            print(f'  [✅] {name}: {df.shape}')
        except Exception as e:
            results[name] = f'ERR: {e}'
            print(f'  [❌] {name}: {e}')
        time.sleep(1.5)
    print(f'\n크롤링 완료: {datetime.now():%Y-%m-%d %H:%M:%S}')
    return results

# ── 자정 자동 스케줄 (운영 환경에서 주석 해제) ─────────────────
# schedule.every().day.at('00:00').do(crawl_all)
# import threading
# def run_scheduler():
#     while True:
#         schedule.run_pending()
#         time.sleep(60)
# t = threading.Thread(target=run_scheduler, daemon=True)
# t.start()
# print("⏰ 자정 자동 업데이트 스케줄 등록 완료")

print("✅ 크롤링 함수 OK")
print("▶  즉시 실행: crawl_all()")
print("▶  자동 스케줄: 위 주석 해제")

## CELL 7 · 탐색적 데이터 분석 (EDA)

In [ ]:
assert 'team_master' in dir(), "CELL 5를 먼저 실행하세요!"

COLS5  = ['#534AB7','#1D9E75','#D85A30','#185FA5','#E85D24']
YR_PAL = dict(zip([2022,2023,2024,2025,2026], COLS5))

# ── 7-1. 리그 환경 5년 트렌드 ────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(16,4))
fig.suptitle('KBO 리그 환경 5년 트렌드 (2022~2026)', fontsize=14, fontweight='bold')
yrs = league_bat['연도'].tolist()

plots = [
    ('AVG',    league_bat, '리그 평균 타율',    (.260,.310)),
    ('ERA',    league_pit, '규정이닝 평균 ERA', (3.0, 6.0)),
    ('K/9',    league_pit, '평균 K/9',          (6.5, 9.5)),
]
for ax,(col,src,title,ylim) in zip(axes,plots):
    vals=src[col].tolist()
    bars=ax.bar(yrs,vals,color=COLS5[:len(yrs)],alpha=.85,edgecolor='white',linewidth=1.2)
    ax.set_title(title,fontsize=11,fontweight='bold'); ax.set_ylim(*ylim)
    for yr,v in zip(yrs,vals):
        ax.text(yr,v+(ylim[1]-ylim[0])*.02,f'{v:.3f}' if col=='AVG' else f'{v:.2f}',
                ha='center',fontsize=9,fontweight='bold')
plt.tight_layout()
plt.savefig('01_league_trend.png',dpi=150,bbox_inches='tight')
plt.show()

# ── 7-2. 팀별 5년 순위 히트맵 ────────────────────────────────
yr_cols = [y for y in [2022,2023,2024,2025,2026] if y in team_master['연도'].values]
pivot = (team_master.pivot_table(index='팀명',columns='연도',values='순위',aggfunc='first')
         [yr_cols].sort_values(yr_cols[-1]))
fig,ax = plt.subplots(figsize=(11,6))
sns.heatmap(pivot,annot=True,fmt='.0f',cmap='RdYlGn_r',vmin=1,vmax=10,
            ax=ax,linewidths=.5,cbar_kws={'label':'순위'},annot_kws={'size':12,'weight':'bold'})
ax.set_title('팀별 연도별 순위 변화 (2022~2026)', fontsize=13,fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout()
plt.savefig('02_rank_heatmap.png',dpi=150,bbox_inches='tight')
plt.show()

# ── 7-3. ERA vs 승률 산점도 ──────────────────────────────────
era_col = next((c for c in team_master.columns if 'ERA' in c and '투수' in c),'ERA_투수')
comp = team_master[team_master['연도']<2026].copy()
if era_col in comp.columns:
    fig,ax = plt.subplots(figsize=(9,6))
    for yr,grp in comp.groupby('연도'):
        sc=ax.scatter(grp[era_col],grp['승률'],c=YR_PAL[yr],label=str(yr),
                      s=120,alpha=.85,edgecolors='white',linewidth=1.5)
        for _,row in grp.iterrows():
            ax.annotate(row['팀명'],(row[era_col],row['승률']),
                        textcoords='offset points',xytext=(5,3),fontsize=8)
    corr=comp[era_col].corr(comp['승률'])
    ax.set_xlabel('팀 ERA',fontsize=11); ax.set_ylabel('승률',fontsize=11)
    ax.set_title(f'팀 ERA vs 승률  (상관계수 r={corr:.3f})',fontsize=13,fontweight='bold')
    ax.legend(title='연도',fontsize=9); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig('03_era_winrate.png',dpi=150,bbox_inches='tight')
    plt.show()

# ── 7-4. 가을야구 진출 vs 미진출 지표 비교 (박스플롯) ─────────
complete = team_master[team_master['가을야구'].notna()].copy()
complete['가을야구_레이블'] = complete['가을야구'].map({1:'✅ 진출',0:'❌ 미진출'})

chk_cols = [c for c in [era_col,
             next((c for c in team_master.columns if 'OPS' in c and '타자' in c),None),
             next((c for c in team_master.columns if 'WHIP' in c and '투수' in c),None),
             '득실차','기대승률'] if c and c in complete.columns]

if chk_cols:
    fig,axes=plt.subplots(1,len(chk_cols),figsize=(4*len(chk_cols),5))
    if len(chk_cols)==1: axes=[axes]
    palette={'✅ 진출':'#1D9E75','❌ 미진출':'#D85A30'}
    for ax,col in zip(axes,chk_cols):
        sns.boxplot(data=complete,x='가을야구_레이블',y=col,palette=palette,ax=ax,width=.5)
        ax.set_title(col.split('_')[0],fontsize=10,fontweight='bold')
        ax.set_xlabel(''); ax.grid(alpha=.3,axis='y')
    fig.suptitle('가을야구 진출/미진출 팀 지표 비교 (2022~2025)',fontsize=13,fontweight='bold')
    plt.tight_layout()
    plt.savefig('04_playoff_boxplot.png',dpi=150,bbox_inches='tight')
    plt.show()

# ── 7-5. 선수이동 현황 (2026) ─────────────────────────────────
if '선수이동현황' in data:
    tr=data['선수이동현황']
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    # 항목별 건수
    cnt=tr['항목'].value_counts().head(8)
    axes[0].barh(cnt.index[::-1],cnt.values[::-1],color='#534AB7',alpha=.85)
    axes[0].set_title('2026 선수이동 항목별 건수',fontsize=11,fontweight='bold')
    axes[0].set_xlabel('건수')
    # 팀별 건수
    t_cnt=tr['팀'].value_counts()
    colors_t=[YR_PAL.get(2024,'#534AB7')]*len(t_cnt)
    axes[1].bar(t_cnt.index,t_cnt.values,color='#D85A30',alpha=.85,edgecolor='white')
    axes[1].set_title('2026 팀별 선수이동 건수',fontsize=11,fontweight='bold')
    axes[1].set_ylabel('건수'); plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('05_transfer.png',dpi=150,bbox_inches='tight')
    plt.show()

## CELL 8 · 가을야구 진출 예측 (앙상블 ML)

In [ ]:
assert 'team_master' in dir(), "CELL 5를 먼저 실행하세요!"

# ── 피처 자동 탐지 ────────────────────────────────────────────
era_col  = next((c for c in team_master.columns if 'ERA'  in c and '투수' in c), None)
ops_col  = next((c for c in team_master.columns if 'OPS'  in c and '타자' in c), None)
whip_col = next((c for c in team_master.columns if 'WHIP' in c and '투수' in c), None)
k9_col   = next((c for c in team_master.columns if 'K/9'  in c and '투수' in c), None)
bb9_col  = next((c for c in team_master.columns if 'BB/9' in c and '투수' in c), None)
fpct_col = next((c for c in team_master.columns if 'FPCT' in c and '수비' in c), None)
sb_col   = next((c for c in team_master.columns if 'SB%'  in c and '주루' in c), None)
rg_col   = next((c for c in team_master.columns if '득점/G' in c), None)
hrg_col  = next((c for c in team_master.columns if 'HR/G' in c), None)

FEATURES = [c for c in [
    '승률', era_col, ops_col, whip_col, k9_col, bb9_col,
    fpct_col, sb_col, rg_col, hrg_col,
    '홈승률','원정승률','득실차','기대승률','승률_운'
] if c and c in team_master.columns]

TARGET = '가을야구'
print(f"사용 피처 ({len(FEATURES)}개):")
for i,f in enumerate(FEATURES): print(f"  {i+1:2d}. {f}")

# ── 학습/테스트 분리 ──────────────────────────────────────────
train_df = team_master[team_master['연도'].isin(COMPLETE_YRS)].dropna(subset=FEATURES+[TARGET]).copy()
test_df  = team_master[team_master['연도']==2026].copy()

X_tr,y_tr = train_df[FEATURES].values, train_df[TARGET].values
scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_tr)

# 2026 예측 데이터 (결측 피처는 훈련 중앙값으로 대체)
medians = train_df[FEATURES].median()
test_filled = test_df[FEATURES].fillna(medians)
X_te_sc = scaler.transform(test_filled.values)

# ── 모델 학습 (Leave-One-Year-Out CV) ────────────────────────
cv_logo = LeaveOneGroupOut()
groups  = train_df['연도'].values
cv_kfold= StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

models = {
    'Logistic': LogisticRegression(C=0.5,max_iter=1000,random_state=42),
    'RF':       RandomForestClassifier(n_estimators=300,max_depth=4,
                                       min_samples_leaf=2,random_state=42),
    'GBM':      GradientBoostingClassifier(n_estimators=200,learning_rate=.05,
                                            max_depth=3,random_state=42),
}

print("\n=== Leave-One-Year-Out 교차검증 ===")
proba_list = []
for name, m in models.items():
    auc_logo = cross_val_score(m, X_tr_sc, y_tr, cv=cv_logo,
                               groups=groups, scoring='roc_auc')
    acc_kf   = cross_val_score(m, X_tr_sc, y_tr, cv=cv_kfold, scoring='accuracy')
    print(f"{name:<12} AUC(LOGO)={auc_logo.mean():.3f}±{auc_logo.std():.3f}  "
          f"ACC(KF)={acc_kf.mean():.3f}±{acc_kf.std():.3f}")
    m.fit(X_tr_sc, y_tr)
    proba_list.append(m.predict_proba(X_te_sc)[:,1])

# ── 앙상블 확률 (평균) ────────────────────────────────────────
prob_ensemble = np.mean(proba_list, axis=0)

# ── 역대 예측 정확도 검증 ─────────────────────────────────────
print("\n=== 역대 연도별 예측 검증 ===")
for yr in COMPLETE_YRS:
    sub = team_master[team_master['연도']==yr].dropna(subset=FEATURES+[TARGET]).copy()
    if len(sub)==0: continue
    X_sub = scaler.transform(sub[FEATURES].fillna(medians).values)
    plist = [m.predict_proba(X_sub)[:,1] for m in models.values()]
    sub['예측확률'] = np.mean(plist,axis=0)
    sub['예측가을야구'] = (sub['예측확률']>=0.5).astype(int)
    acc = (sub['예측가을야구']==sub['가을야구']).mean()
    auc = roc_auc_score(sub['가을야구'],sub['예측확률']) if sub['가을야구'].nunique()>1 else np.nan
    print(f"  {yr}년: ACC={acc:.1%}  AUC={auc:.3f}  "
          f"진출예측: {sub[sub['예측가을야구']==1]['팀명'].tolist()}")

# ── 2026 예측 결과 ────────────────────────────────────────────
result = test_df[['팀명','순위','승','패','승률']].copy()
if era_col  in test_df: result['ERA']  = test_df[era_col].values
if ops_col  in test_df: result['OPS']  = test_df[ops_col].values
if whip_col in test_df: result['WHIP'] = test_df[whip_col].values
result['가을야구_확률(%)'] = (prob_ensemble*100).round(1)
result['예측'] = ['✅ 진출' if p>=0.5 else '❌ 미진출' for p in prob_ensemble]
result = result.sort_values('가을야구_확률(%)',ascending=False).reset_index(drop=True)
result.index += 1
print("\n=== 2026 가을야구 진출 예측 ===")
display(result)

# ── 피처 중요도 ───────────────────────────────────────────────
imp = pd.Series(models['RF'].feature_importances_, index=FEATURES).sort_values(ascending=True)
fig,ax = plt.subplots(figsize=(9,5))
colors_i=['#1D9E75' if v>.12 else '#534AB7' if v>.06 else '#B4B2A9' for v in imp.values]
imp.plot(kind='barh',ax=ax,color=colors_i,edgecolor='white',linewidth=1.2)
ax.axvline(.1,color='red',ls='--',alpha=.5,label='10% 기준선')
ax.set_title('가을야구 예측 피처 중요도 (Random Forest)',fontsize=12,fontweight='bold')
ax.legend(); ax.grid(alpha=.3,axis='x')
plt.tight_layout()
plt.savefig('06_feature_importance.png',dpi=150,bbox_inches='tight')
plt.show()

# ── 확률 바 차트 ───────────────────────────────────────────────
fig,ax = plt.subplots(figsize=(11,5))
colors_bar=['#1D9E75' if p>=50 else '#D85A30' for p in result['가을야구_확률(%)']]
bars=ax.bar(result['팀명'],result['가을야구_확률(%)'],color=colors_bar,
            edgecolor='white',linewidth=1.5,alpha=.9)
ax.axhline(50,color='gray',ls='--',alpha=.7,label='50% 기준')
for bar,v in zip(bars,result['가을야구_확률(%)']):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f'{v}%',
            ha='center',fontsize=11,fontweight='bold')
ax.set_title('2026 KBO 가을야구 진출 확률 예측',fontsize=14,fontweight='bold')
ax.set_ylabel('진출 확률 (%)'); ax.set_ylim(0,115)
ax.legend(fontsize=10); ax.grid(alpha=.3,axis='y')
plt.tight_layout()
plt.savefig('07_playoff_prob.png',dpi=150,bbox_inches='tight')
plt.show()

## CELL 9 · 심층 분석 — 4월 예측력 · 팀 클러스터링 · 선수이동 영향

In [ ]:
assert 'team_master' in dir(), "CELL 5를 먼저 실행하세요!"

# ── 9-1. 4월 순위 → 최종순위 예측력 ─────────────────────────
apr = (daily[(daily['연도']<2026) & (daily['월']==4)]
       .groupby(['연도','팀명'])['순위'].mean()
       .reset_index().rename(columns={'순위':'4월평균순위'}))
mg  = apr.merge(
    team_master[team_master['연도']<2026][['연도','팀명','순위','가을야구']],
    on=['연도','팀명'])
corr_apr = mg['4월평균순위'].corr(mg['순위'])
acc_apr  = ((mg['4월평균순위']<=5) == mg['가을야구'].astype(bool)).mean()

fig,ax = plt.subplots(figsize=(7,6))
pal4 = {2022:'#534AB7',2023:'#1D9E75',2024:'#D85A30',2025:'#185FA5'}
for yr,grp in mg.groupby('연도'):
    ax.scatter(grp['4월평균순위'],grp['순위'],c=pal4[yr],label=str(yr),s=90,alpha=.85,edgecolors='white')
    for _,row in grp.iterrows():
        ax.annotate(row['팀명'],(row['4월평균순위'],row['순위']),
                    textcoords='offset points',xytext=(4,2),fontsize=7)
ax.plot([1,10],[1,10],'k--',alpha=.3,label='y=x')
ax.set(xlabel='4월 평균순위',ylabel='최종 순위',
       title=f'4월 순위 vs 최종순위 (r={corr_apr:.3f})')
ax.legend(title='연도',fontsize=9); ax.grid(alpha=.3)
ax.invert_xaxis(); ax.invert_yaxis()
plt.tight_layout()
plt.savefig('08_apr_vs_final.png',dpi=150,bbox_inches='tight')
plt.show()
print(f"📊 4월↔최종 상관계수: {corr_apr:.3f}")
print(f"📊 4월 순위 단독 가을야구 예측 정확도: {acc_apr:.1%}  (ML 앙상블과 비교)")

# 2026 4월 현재 예측
apr_2026 = (daily[(daily['연도']==2026) & (daily['월']==4)]
            .groupby('팀명')['순위'].mean()
            .reset_index().rename(columns={'순위':'4월평균순위'})
            .sort_values('4월평균순위'))
print("\n=== 2026 4월 평균순위 (현재 페이스) ===")
display(apr_2026)

# ── 9-2. 팀 클러스터링 (K-Means) ─────────────────────────────
fc_all   = [c for c in [era_col,ops_col,whip_col,'홈승률','원정승률',rg_col]
            if c and c in team_master.columns]
cl_base  = team_master[team_master['연도']<2026].dropna(subset=fc_all).copy()
sc_cl    = StandardScaler()
X_cl     = sc_cl.fit_transform(cl_base[fc_all])

km = KMeans(n_clusters=3,random_state=42,n_init=10)
cl_base['클러스터'] = km.fit_predict(X_cl)

rnk_mean = cl_base.groupby('클러스터')['순위'].mean()
nm_map   = {rnk_mean.idxmin():'🏆 강팀형', rnk_mean.idxmax():'📉 약팀형'}
for k in cl_base['클러스터'].unique():
    if k not in nm_map: nm_map[k]='📊 중위권형'
cl_base['팀유형'] = cl_base['클러스터'].map(nm_map)

# PCA 시각화
pca  = PCA(n_components=2,random_state=42)
X_pca= pca.fit_transform(X_cl)
cc   = {'🏆 강팀형':'#1D9E75','📊 중위권형':'#534AB7','📉 약팀형':'#D85A30'}

fig,ax = plt.subplots(figsize=(10,7))
for ct,grp in cl_base.groupby('팀유형'):
    idx=[list(cl_base.index).index(i) for i in grp.index]
    ax.scatter(X_pca[idx,0],X_pca[idx,1],c=cc[ct],label=ct,s=100,alpha=.85,edgecolors='white')
    for i,(_,row) in zip(idx,grp.iterrows()):
        ax.annotate(f"{row['팀명']}({row['연도']})",
                    (X_pca[i,0],X_pca[i,1]),textcoords='offset points',
                    xytext=(5,3),fontsize=7)
ax.set_title(f'팀 유형 클러스터링 PCA (2022~2025)\nPC1 설명분산: {pca.explained_variance_ratio_[0]:.1%}',
             fontsize=12,fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig('09_cluster_pca.png',dpi=150,bbox_inches='tight')
plt.show()

print("\n=== 클러스터별 평균 지표 ===")
display(cl_base.groupby('팀유형')[fc_all+['순위','승률']].mean().round(3))

# 2026 팀 유형 분류
if len(test_df.dropna(subset=fc_all))>0:
    X_26  = sc_cl.transform(test_df[fc_all].fillna(cl_base[fc_all].median()).values)
    test_df['팀유형'] = [nm_map[l] for l in km.predict(X_26)]
    print("\n=== 2026 팀 유형 분류 ===")
    display(test_df[['팀명','순위','팀유형']].sort_values('순위'))

# ── 9-3. 선수이동 영향 분석 ──────────────────────────────────
if '선수이동현황' in data:
    tr=data['선수이동현황'].copy()
    # 부상자 명단 누적 팀별
    injured = (tr[tr['항목'].isin(['부상자 명단','치료·재활명단'])]
               .groupby('팀').size().reset_index(name='부상_누적건수')
               .sort_values('부상_누적건수',ascending=False))
    print("\n=== 2026 팀별 부상 누적 건수 ===")
    display(injured)
    
    # FA/트레이드로 이탈한 주요 선수
    fa_trade = tr[tr['항목'].isin(['자유계약선수','트레이드','FA 계약'])]
    print(f"\n=== 2026 이적/FA 현황 ({len(fa_trade)}건) ===")
    display(fa_trade[['날짜','항목','팀','선수','비고']].head(15))

## CELL 10 · 최종 결론 출력

In [ ]:
assert 'result' in dir(), "CELL 8을 먼저 실행하세요!"

print('='*65)
print('  KBO 2026 가을야구 진출 예측 — 최종 결과')
print(f'  기준일: 2026-04-24  |  팀당 약 22경기 진행')
print(f'  학습 데이터: 2022~2025 ({len(COMPLETE_YRS)}개 시즌, 40개 팀×시즌)')
print('='*65)

진출 = result[result['예측']=='✅ 진출']
미진출= result[result['예측']=='❌ 미진출']

print('\n  ✅ 가을야구 진출 예측 팀:')
for _,r in 진출.iterrows():
    era_str = f"ERA {r['ERA']:.2f}" if 'ERA' in r.index and pd.notna(r.get('ERA')) else ''
    ops_str = f"OPS {r['OPS']:.3f}" if 'OPS' in r.index and pd.notna(r.get('OPS')) else ''
    print(f"    {r['팀명']:5s}  확률 {r['가을야구_확률(%)']:5.1f}%  "
          f"(현재 {int(r['순위'])}위 {int(r['승'])}승 {int(r['패'])}패  {era_str} {ops_str})")

print('\n  ❌ 미진출 예측 팀:')
for _,r in 미진출.iterrows():
    print(f"    {r['팀명']:5s}  확률 {r['가을야구_확률(%)']:5.1f}%  (현재 {int(r['순위'])}위)")

print('\n  📊 핵심 발견 (2022~2025 데이터 기반):')
fi=pd.Series(models['RF'].feature_importances_,index=FEATURES).sort_values(ascending=False)
top3=fi.head(3)
for i,(nm,v) in enumerate(top3.items(),1):
    print(f'    {i}. {nm}  →  중요도 {v:.1%}')

print(f'\n  📈 4월 순위 단독 예측 정확도:  {acc_apr:.1%}')
print(f'  📈 ML 앙상블 역대 검증 정확도:  ~{(sum([(team_master[team_master["연도"]==yr].dropna(subset=FEATURES+[TARGET]).shape[0]) for yr in COMPLETE_YRS if yr in team_master["연도"].values]>0)*100):.0f}% (연도별 결과 참조)')
print(f'\n  ⚠️  주의: {len(진출)}팀 예측이 5팀 미만/초과인 경우 확률 0.5 근접 팀 주의')
print('='*65)

# 저장
result.to_csv('2026_가을야구_예측결과.csv',index=True,encoding='utf-8-sig')
team_master.to_csv('team_master_2022_2026.csv',index=False,encoding='utf-8-sig')
print('\n💾 결과 CSV 저장 완료:')
print('   - 2026_가을야구_예측결과.csv')
print('   - team_master_2022_2026.csv')